In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/clean_brick_dataset_v3.csv')
print(df.shape)
df.head()

(355, 15)


,Districts,Location,Company Name,Company Type,Machine Type,Product Type,Block Type,Company Category,Company Size,Contact,Website,Map,Status,has_website,has_contact
0,Medak,"Ambjipet, Shankarampet (R), Medak, Telangana 5...",Srinivas Brick suppliers,Manufacturing,Hand Made,Fired Clay Brick,Standard Block,Traditional Clay Factory,Large,9849996544,NaN,https://maps.app.goo.gl/hjq2dTVZJzQehqQ96,Running,False,True
1,Medak,"Shalipet, Telangana 502113, India",DMi bricks medak,NaN,Unknown,Unknown,Unknown,Traditional Clay Factory,Unknown,NaN,NaN,NaN,Permanently Closed,False,False
2,Medak,"Roopla thanda, shivampet(mdl, Narsapur, Telang...",SRI LAXMI MRK BRICKS,Manufacturing,Hand Made,Fired Clay Brick,Standard Block,Traditional Clay Factory,Moderate,9989836762,NaN,https://maps.app.goo.gl/g7GCC8aVLbjEZiX96,Running,False,True
3,Medak,"Road, Markandeya Nagar Colony, Nizampur, Sadas...",KMB Bricks Field,Manufacturing,Hand Made,Fired Clay Brick,Standard Block,Traditional Clay Factory,Large,9440030136,NaN,https://maps.app.goo.gl/qkcxuFxH8BPH85zd6,Running,False,True
4,Medak,"HATNOORA, VILLAGE, NAVABPETA, ROAD, Sangareddy...",Bondada ECOBuild Pvt Ltd - AAC Block Division,Manufacturing,Hand Made,Fly Ash Brick,Standard Block,Fully Automatic Fly Ash Brick Plant,Large,NaN,https://bondadaecobuild.com/,https://maps.app.goo.gl/3My3HpDDDoDJWouJ6,Running,True,False


# 02 — KPI Analysis
Market-level and district-level KPIs from the cleaned dataset.

In [2]:
kpis = {
    'total_factories': len(df),
    'running': (df['Status'] == 'Running').sum(),
    'permanently_closed': (df['Status'] == 'Permanently Closed').sum(),
    'temporarily_closed': (df['Status'] == 'Temporarily Closed').sum(),
    'operational_rate_pct': round((df['Status'] == 'Running').mean() * 100, 1),
    'eco_share_pct': round((df['Product Type'] == 'Fly Ash Brick').mean() * 100, 1),
    'handmade_share_pct': round((df['Machine Type'] == 'Hand Made').mean() * 100, 1),
    'fully_automatic_pct': round((df['Machine Type'] == 'Fully Automatic Machine Made').mean() * 100, 1),
    'has_website_pct': round(df['has_website'].mean() * 100, 1),
    'has_contact_pct': round(df['has_contact'].mean() * 100, 1),
}

kpi_df = pd.DataFrame(kpis.items(), columns=['KPI', 'Value'])
kpi_df

,KPI,Value
0,total_factories,355.0
1,running,303.0
2,permanently_closed,26.0
3,temporarily_closed,26.0
4,operational_rate_pct,85.4
5,eco_share_pct,65.9
6,handmade_share_pct,55.8
7,fully_automatic_pct,18.0
8,has_website_pct,13.2
9,has_contact_pct,57.7


In [3]:
district = df.groupby('Districts').agg(
    factory_count=('Company Name', 'count'),
    running_count=('Status', lambda x: (x == 'Running').sum()),
    eco_count=('Product Type', lambda x: (x == 'Fly Ash Brick').sum()),
    fully_auto_count=('Machine Type', lambda x: (x == 'Fully Automatic Machine Made').sum()),
    closure_count=('Status', lambda x: (x != 'Running').sum()),
).reset_index()

district['operational_rate'] = (district['running_count'] / district['factory_count'] * 100).round(1)
district['eco_share'] = (district['eco_count'] / district['factory_count'] * 100).round(1)
district['closure_rate'] = (district['closure_count'] / district['factory_count'] * 100).round(1)

district.sort_values('factory_count', ascending=False)

,Districts,factory_count,running_count,eco_count,fully_auto_count,closure_count,operational_rate,eco_share,closure_rate
15,Medchal Malkajgiri,42,41,32,10,1,97.6,76.2,2.4
3,Hyderabad,30,28,26,13,2,93.3,86.7,6.7
30,Yadadri Bhuvanagiri,27,23,16,4,4,85.2,59.3,14.8
2,Hanumakonda,22,19,19,2,3,86.4,86.4,13.6
22,Peddapalli,21,20,7,1,1,95.2,33.3,4.8
9,Karimnagar,20,20,10,3,0,100.0,50.0,0.0
13,Mancherial,16,13,11,3,3,81.2,68.8,18.8
5,Jangaon,14,13,11,1,1,92.9,78.6,7.1
18,Nalgonda,13,12,9,2,1,92.3,69.2,7.7
25,Sangareddy,11,11,2,2,0,100.0,18.2,0.0


In [4]:
kpi_df.to_csv('../data/processed/brick_kpi_summary.csv', index=False)
district.to_csv('../data/processed/brick_district_summary.csv', index=False)
print("KPI files exported ✓")

KPI files exported ✓
